In [7]:
from pathlib import Path
import sys
sys.dont_write_bytecode = True

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

mpl.rcParams.update({
    "font.family": "serif",
    "font.size": 14,
    "axes.labelsize": 16,
    "axes.titlesize": 14,
    "legend.fontsize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "lines.linewidth": 1.4,
    "axes.linewidth": 1,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
})

cwd = Path.cwd()
if (cwd / "utils.py").exists() and (cwd / "data").exists():
    notebook_dir = cwd
elif (cwd / "charge_stability_Leburton" / "utils.py").exists():
    notebook_dir = cwd / "charge_stability_Leburton"
else:
    notebook_dir = Path("charge_stability_Leburton").resolve()

if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

from utils import DataHelper

data_dir = notebook_dir / "data"


In [8]:
data_helper = DataHelper(
    data_dir=data_dir,
    results_filename="charge_stability.pkl",
    checkpoint_glob="charge_stability_runs_*.pkl",
)

checkpoint_path = data_helper.newest_checkpoint_path()
checkpoint_paths = [checkpoint_path] if checkpoint_path is not None else []
loaded_data = data_helper.load_data_with_checkpoints(
    default={"metadata": {}, "runs": [], "charge_stability": {}},
    checkpoint_paths=checkpoint_paths,
)

full_results = loaded_data["data"] or {"metadata": {}, "runs": [], "charge_stability": {}}
checkpoint_records = loaded_data["checkpoint_records"]
checkpoint_headers = [record for record in checkpoint_records if record.get("kind") == "header"]
checkpoint_run_records = [record for record in checkpoint_records if record.get("kind") == "run"]

metadata = dict(full_results.get("metadata", {}))
sweep_config = checkpoint_headers[-1].get("sweep_config", {}) if checkpoint_headers else {}
sweep_hash = checkpoint_headers[-1].get("sweep_hash") if checkpoint_headers else metadata.get("checkpoint_sweep_hash")

print(f"data directory: {data_dir}")
print(f"full results: {data_helper.results_path} ({'found' if data_helper.results_path.exists() else 'missing'})")
print(f"checkpoint: {checkpoint_path}")
print(f"checkpoint records: {len(checkpoint_records)} total, {len(checkpoint_run_records)} run(s)")
print(f"sweep hash: {sweep_hash}")


data directory: /home/leander/Documents/PhD/Research_Projects/QDots/charge_stability_Leburton/data
full results: /home/leander/Documents/PhD/Research_Projects/QDots/charge_stability_Leburton/data/charge_stability.pkl (missing)
checkpoint: /home/leander/Documents/PhD/Research_Projects/QDots/charge_stability_Leburton/data/charge_stability_runs_476cd5197895.pkl
checkpoint records: 5 total, 4 run(s)
sweep hash: 476cd5197895


In [9]:
checkpoint_runs_by_grid_point = {
    (int(record["i"]), int(record["j"])): record["run"]
    for record in checkpoint_run_records
}

runs_by_grid_point = dict(checkpoint_runs_by_grid_point)

if not runs_by_grid_point:
    voltages = metadata.get("voltage_grid", sweep_config.get("voltages", []))
    voltage_grid = np.asarray(voltages, dtype=float)

    def grid_key_from_run(run):
        i = int(np.argmin(np.abs(voltage_grid - float(run["vL"]))))
        j = int(np.argmin(np.abs(voltage_grid - float(run["vR"]))))
        return i, j

    runs_by_grid_point = {
        grid_key_from_run(run): run
        for run in full_results.get("runs", [])
    }

runs = [runs_by_grid_point[key] for key in sorted(runs_by_grid_point)]
charge_stability = {
    "metadata": metadata,
    "sweep_config": sweep_config,
    "sweep_hash": sweep_hash,
    "runs": runs,
    "runs_by_grid_point": runs_by_grid_point,
}

print(f"loaded completed points: {len(runs_by_grid_point)}")
for key, run in sorted(runs_by_grid_point.items()):
    print(f"grid {key}: vL={1e3 * run['vL']:.3f} mV, vR={1e3 * run['vR']:.3f} mV, stable n={run.get('n_electrons')}")


loaded completed points: 4
grid (0, 0): vL=21.000 mV, vR=21.000 mV, stable n=1
grid (1, 0): vL=21.889 mV, vR=21.000 mV, stable n=2
grid (1, 1): vL=21.889 mV, vR=21.889 mV, stable n=2
grid (2, 0): vL=22.778 mV, vR=21.000 mV, stable n=2


In [10]:
def result_with_stored_potential(runs_by_grid_point):
    for grid_key, run in sorted(runs_by_grid_point.items()):
        for n_electrons, result in sorted(run.get("points", {}).items(), key=lambda item: int(item[0])):
            potential = result.get("potential")
            if potential is not None:
                return grid_key, run, int(n_electrons), result, np.asarray(potential, dtype=float), "stored"
    return None


def analytic_potential_line(run, sweep_config, n_points=2001):
    L = float(sweep_config.get("L", 1.0))
    R = float(sweep_config["R"])
    d = float(sweep_config["d"])
    half_width = 10.0 * L
    x_centered = np.linspace(-half_width, half_width, n_points)
    v_l = float(run["vL"])
    v_r = float(run["vR"])
    potential = (
        -v_l * np.exp(-((x_centered + d / 2) ** 2) / (R**2))
        -v_r * np.exp(-((x_centered - d / 2) ** 2) / (R**2))
    )
    return np.column_stack([x_centered, potential])


entry = result_with_stored_potential(runs_by_grid_point)
if entry is None:
    if not runs:
        raise ValueError("No completed run was found in the checkpoint or full results.")
    if not {"L", "R", "d"}.issubset(sweep_config):
        raise ValueError("No stored potential found, and sweep_config is missing L/R/d for analytic reconstruction.")
    grid_key = sorted(runs_by_grid_point)[0]
    run = runs_by_grid_point[grid_key]
    n_electrons = run.get("n_electrons")
    result = None
    potential_data = analytic_potential_line(run, sweep_config)
    source = "analytic"
else:
    grid_key, run, n_electrons, result, potential_data, source = entry

x_raw = potential_data[:, 0]
potential = potential_data[:, 1]
if source == "stored":
    x = x_raw - 0.5 * (x_raw[0] + x_raw[-1])
else:
    x = x_raw

print(f"potential source: {source}")
print(f"grid point: {grid_key}")
print(f"voltages: vL={1e3 * run['vL']:.3f} mV, vR={1e3 * run['vR']:.3f} mV")
print(f"electron point: n={n_electrons}")
print(f"potential samples: {len(potential)}")


potential source: stored
grid point: (0, 0)
voltages: vL=21.000 mV, vR=21.000 mV
electron point: n=1
potential samples: 2001
